# Diabetes Prevalence Analysis
## Pipeline Results

SVM regression and Belief Network models trained on WHO + World Bank data.  
Simulations projected to 2030, 2040, 2050 across 10 countries and 7 scenarios.

In [ ]:
import joblib
import pandas as pd
from IPython.display import Image, display
from pathlib import Path

## SVM Model
Trained with HalvingGridSearchCV on historical WHO + World Bank data.  
Test set: 2014–2016 holdout.

In [ ]:
metrics = joblib.load("../outputs/models/svm_metrics.pkl")
pi      = joblib.load("../outputs/models/svm_permutation_importance.pkl")
bounds  = joblib.load("../outputs/models/svm_feature_bounds.pkl")

print("SVM Metrics (test set 2014–2016)")
for k, v in metrics.items():
    print(f"  {k.upper():<6}: {v:.4f}")

print("\nTop Features by Permutation Importance:")
importance_df = pd.DataFrame({
    "feature":    bounds.index,
    "importance": pi.importances_mean,
}).sort_values("importance", ascending=False)
print(importance_df.to_string(index=False))

In [ ]:
for path in [
    "../outputs/figures/svm_actual_vs_predicted.png",
    "../outputs/figures/svm_feature_importance.png",
    "../outputs/figures/svm_residual_analysis.png",
    "../outputs/figures/svm_partial_dependence.png",
]:
    display(Image(filename=path))

## Belief Network
Trained on discretized features (low / medium / high bins).  
Shows causal structure and conditional probabilities.

In [ ]:
bn_metrics = joblib.load("../outputs/models/bn_metrics.pkl")

print("Belief Network Metrics (test set 2014–2016)")
for k, v in bn_metrics.items():
    print(f"  {k.upper():<6}: {v:.4f}")

bn_queries = joblib.load("../outputs/models/bn_causal_queries.pkl")

rows = []
for feature, levels in bn_queries.items():
    delta = levels["high"]["high"] - levels["low"]["high"]
    rows.append({"feature": feature, "delta_P(diabetes=high)": round(delta, 4)})

delta_df = pd.DataFrame(rows).sort_values("delta_P(diabetes=high)", ascending=False)
print("\nP(diabetes=high | feature=high)  −  P(diabetes=high | feature=low)")
print(delta_df.to_string(index=False))

In [ ]:
for path in [
    "../outputs/figures/bn_network_graph.png",
    "../outputs/figures/bn_cpd_diabetes.png",
]:
    display(Image(filename=path))

## Simulations
7 scenarios projected to 2030, 2040, 2050 across 10 countries.

In [ ]:
sim_df = pd.read_csv("../outputs/reports/simulation_results.csv")

pivot = (
    sim_df[sim_df["year"] == 2050]
    .pivot(index="iso3_code", columns="scenario", values="predicted_prevalence")
    .round(2)
)
print("Predicted Diabetes Prevalence in 2050 (%) by Country and Scenario")
display(pivot)

In [ ]:
for path in [
    "../outputs/figures/sim_scenario_comparison.png",
    "../outputs/figures/sim_cross_country_heatmap.png",
    "../outputs/figures/sim_intervention_deltas.png",
    "../outputs/figures/sim_timeline_IND.png",
    "../outputs/figures/sim_timeline_USA.png",
    "../outputs/figures/sim_timeline_CHN.png",
]:
    display(Image(filename=path))

## Pipeline Report

In [ ]:
print(open("../outputs/reports/pipeline_report.txt").read())